# AI Co-Scientist: Blind Data and Hypotheses

* * * 

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Describe the four stages of an AI-assisted research workflow.
* Explain why the data are disguised before an AI model sees them.
* Spot a power law by plotting noisy data on log-log axes.
* Run the Co-Scientist agent loop by hand in a chat app, or with code.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
🥊 **Challenge**: Interactive exercise. We'll work through these in the workshop!<br>
⏱️ **Take-home**: A challenge we skip in the 1-hour workshop. Try it afterwards; answers are in `solutions/`.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
🎬 **Demo**: Showing off something more advanced – so you know what Python can be used for!<br>

### Sections
1. [The Research Workflow](#workflow)
2. [The Blind Dataset](#data)
3. [Looking for a Pattern](#pattern)
4. [Generating Hypotheses With Agents](#agents)
5. [Demo: Open Coscientist](#demo)

<a id='workflow'></a>

# The Research Workflow

In this workshop, AI tools help with each step of a small research project:
propose hypotheses, gather evidence, test the hypotheses, and write the paper.

| Part | Question | AI tool | No-key fallback |
| --- | --- | --- | --- |
| 1. Hypotheses | What could the law be? | [Open Coscientist](https://github.com/jataware/open-coscientist) | Prompts by hand in a chat app |
| 2. Evidence | Which formula predicts unseen groups? | [OpenEvolve](https://github.com/algorithmicsuperintelligence/openevolve) | `mini_evolve.py` |
| 3. Tests | Is the evidence strong enough? | None: plain statistics | Always offline |
| 4. Paper | How does it compare with the literature? | [OpenScience](https://github.com/synthetic-sciences/openscience) | `make_paper.py` |

The task: given 29 noisy measurements of three unnamed quantities, `x1`, `x2`
and `y`, find the **law** that links them, and show that it holds for groups
it never saw.

Every part runs without an API key. Cells that call an AI model check for a
key first and skip themselves if there is none.

In [ ]:
# Imports and paths used in this notebook.
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'lessons' / 'scripts' / '02_openevolve'))
from evaluator import load_rows, log_error

<a id='data'></a>

# The Blind Dataset

The data live in `data/observations.csv`. Each row has:

- `group`: rows in the same group come from the same system.
- `x1`, `x2`, `y`: three measured quantities, in arbitrary units.
- `split`: `train` rows are for finding the law; `test` rows are for checking it.

The test rows come from **whole groups** that are held out, so a law only
scores well on them if it works for a system it has never seen.

In [ ]:
rows = load_rows(ROOT / 'data' / 'observations.csv')
train = [row for row in rows if row['split'] == 'train']
test = [row for row in rows if row['split'] == 'test']
print(len(rows), 'rows:', len(train), 'train and', len(test), 'test')

In [ ]:
for row in rows[:5]:
    print(row)

In [ ]:
groups = sorted({row['group'] for row in rows})
for group in groups:
    in_group = [row for row in rows if row['group'] == group]
    print(f"group {group}: {len(in_group)} rows, x2 = {in_group[0]['x2']:.4g}, split = {in_group[0]['split']}")

🔔 **Question:** Every row in a group has the same `x2`. What does that tell
you about what `x2` might describe?

## The Disguise

The numbers are real measurements, but `data/make_dataset.py` disguised them
before any AI model saw them:

- **Names are hidden.** Systems became groups `A` to `G`, and columns became `x1`, `x2`, `y`.
- **Units are scrambled.** A constant remembered from a textbook will not fit.
- **Noise is added.** The law has to be fitted, not read off.

Why bother? A language model has read most of science. If it is told what the
numbers are, it can **recite** a famous law instead of **discovering** it from
the data. In our tests, with real names and units, a model wrote down the
right law, with the textbook constant, on its first try.

⚠️ **Warning:** Do not open `data/orbits.csv` or `data/key.json` yet. They
give the answer away. We open the key at the end of Part 3.

<a id='pattern'></a>

# Looking for a Pattern

Many laws in science are **power laws**: `y = c * x1^p`. Take the logarithm of
both sides and you get a straight line:

`log(y) = log(c) + p * log(x1)`

So on log-log axes, a power law is a straight line, and its slope is the
exponent `p`. Let's plot `y` against `x1`, one color per group.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for group in groups:
    in_group = [row for row in rows if row['group'] == group]
    ax.scatter([row['x1'] for row in in_group], [row['y'] for row in in_group], label=group)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('x1')
ax.set_ylabel('y')
ax.legend(title='group')
plt.show()

🔔 **Question:** Do the groups fall on one line, or on several parallel lines?
If several, what could shift a line up or down?

## 🥊 Challenge 1: The Slope Within One Group

⏱️ **Take-home:** we skip this challenge in the 1-hour workshop. Try it afterwards.

Fit a straight line to `log10(y)` against `log10(x1)` for group `A` only. The
slope is the exponent `p` in `y = c * x1^p`.

Use `np.polyfit(x, y, 1)`, which returns the slope and the intercept of the
best straight line.

In [ ]:
group_a = [row for row in rows if row['group'] == 'A']
log_x1 = np.log10([row['x1'] for row in group_a])
log_y = np.log10([row['y'] for row in group_a])

# YOUR CODE HERE

💡 **Tip:** `np.polyfit()` returns the coefficients from the highest power
down, so for a straight line you get the slope first, then the intercept.

## 🥊 Challenge 2: Score a Hypothesis

A hypothesis is only useful if we can score it. Test the hypothesis
`y = c * x1^1.5`, which ignores `x2`:

1. Fit `c` on the training rows. In log space, `log10(c)` is the mean of
   `log10(y) - 1.5 * log10(x1)`.
2. Fill in `predict()`.
3. `log_error()` gives the mean error in `log10(y)`, in **dex**. An error of
   1 dex means off by a factor of 10.

Both errors are large, even on the training rows. Why can one constant `c`
not fit every group?

In [ ]:
log_c = ...


def predict(x1, x2):
    return ...


print('train error:', round(log_error(predict, train), 3), 'dex')
print('test error: ', round(log_error(predict, test), 3), 'dex')

<a id='agents'></a>

# Generating Hypotheses With Agents

Google's [AI co-scientist](https://research.google/blog/accelerating-scientific-breakthroughs-with-an-ai-co-scientist/)
splits research into a team of agents, each a language model with its own job:

- **Generation:** proposes hypotheses.
- **Reflection:** reviews them like a strict peer reviewer.
- **Ranking:** runs a tournament of debates and keeps an **Elo** score, as in chess.
- **Evolution:** combines and improves the best ones.
- **Meta-review:** summarizes what the team learned.

You can run this loop in two ways:

- **By hand, no key:** paste the prompts in
  [`scripts/01_hypotheses/prompts.md`](scripts/01_hypotheses/prompts.md)
  into a chat app, such as Gemini with your berkeley.edu account. You play the supervisor.
- **With code and a key:** [Open Coscientist](https://github.com/jataware/open-coscientist)
  runs every agent for you. See the demo at the end of this notebook.

Either way, the agents only see the training rows, printed below.

In [ ]:
print('group,x1,x2,y')
for row in train:
    print(f"{row['group']},{row['x1']:.4g},{row['x2']:.4g},{row['y']:.4g}")

## 🥊 Challenge 3: Run the Agent Loop by Hand

⏱️ **Take-home:** we skip this challenge in the 1-hour workshop. Try it afterwards.

1. Open [`scripts/01_hypotheses/prompts.md`](scripts/01_hypotheses/prompts.md).
2. Paste the Generation, Reflection and Ranking prompts into a chat app, one
   at a time, in one conversation. Paste the training rows above where it says
   `<paste the training rows>`.
3. Write the top-ranked hypothesis as `predict()` below, with its constants
   fitted on `train`, and score it with `log_error()` as in Challenge 2.

Does the hypothesis the agents ranked first also have the lowest test error?

In [ ]:
# YOUR CODE HERE

<a id='demo'></a>

# 🎬 Demo: Open Coscientist

With an API key, [Open Coscientist](https://github.com/jataware/open-coscientist)
runs the whole agent team in about two minutes. Set one key before you start
Jupyter, for example `export OPENAI_API_KEY=your-key`. Other providers are
listed in the README.

The cell below skips itself when no key is set.

In [ ]:
if os.environ.get('OPENAI_API_KEY'):
    !{sys.executable} "{ROOT}/lessons/scripts/01_hypotheses/run_coscientist.py" --model gpt-4.1-mini
elif os.environ.get('GEMINI_API_KEY'):
    !{sys.executable} "{ROOT}/lessons/scripts/01_hypotheses/run_coscientist.py" --model gemini/gemini-2.5-flash
else:
    print('No API key set: skipping. Use the prompts in scripts/01_hypotheses/prompts.md instead.')

In our single test run with `gpt-4.1-mini`, the debate ranked first power
laws with the **wrong sign** on the `x2` exponent (+0.5; the data say −0.5).
Your run may differ. Ranking by debate is only as good as the judge. In Part 2,
we replace the judge with a number computed from the data.

<div class="alert alert-success">

## ❗ Key Points

* An AI co-scientist splits research into agents that generate, review, rank and refine hypotheses.
* Disguise the data so the model has to discover the law, not recite it from memory.
* On log-log axes a power law is a straight line, and its slope is the exponent.
* Hold out whole groups, so a law has to work for systems it has never seen.
* A hypothesis ranked first by debate can still be wrong (it was in our run): score it against data.

</div>